In [1]:
# -*- coding: utf-8 -*-
"""
Yukawa Gravity Toolkit: Comprehensive framework for testing scale-dependent spacetime geometries

Original work: Facundo Firmenich et al.
Extended with: ppE mappings, Bayesian analysis, systematic error modeling, and stability analysis
"""

from dataclasses import dataclass
from typing import Tuple, Callable, Dict, List, Optional
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.ndimage import uniform_filter1d
from scipy.optimize import minimize
import warnings

# -------------------------
# Physical constants (IAU/CODATA)
# -------------------------

G0 = 6.67430e-11        # m^3 kg^-1 s^-2
c0 = 2.99792458e8       # m/s
M_SUN = 1.98847e30      # kg
AU = 1.495978707e11     # m
R_SUN = 6.957e8         # m
J2_SUN = 2.25e-7        # Solar quadrupole moment

# -------------------------
# Yukawa parameters class
# -------------------------

@dataclass
class YukawaParams:
    alpha: float    # dimensionless strength α_Y
    lambda_m: float # range λ_Y in meters

# -------------------------
# Power-law ansatz class
# -------------------------

@dataclass
class PowerLawParams:
    epsilon: float  # dimensionless amplitude
    beta: float     # exponent
    ell_s: float    # reference scale [m]

# -------------------------
# RG-improved ansatz class
# -------------------------

@dataclass
class RGImprovedParams:
    xi: float       # amplitude of running
    n: float        # fall-off exponent
    r0: float       # transition scale [m]

# -------------------------
# Basic potential and derivatives
# -------------------------

def phi_yukawa(r: float, M: float, params: YukawaParams) -> float:
    """
    Gravitational potential with Yukawa correction:
        Phi(r) = - G M / r * (1 + alpha * exp(-r/lambda))
    """
    if r <= 0.0:
        raise ValueError(f"r must be > 0, got {r}")
    return -G0 * M / r * (1.0 + params.alpha * np.exp(-r / params.lambda_m))

def phi_power_law(r: float, M: float, params: PowerLawParams) -> float:
    """
    Power-law modified potential:
        Phi(r) = - G M / r * [1 + epsilon * (r/ell_s)^beta]
    """
    if r <= 0.0:
        raise ValueError(f"r must be > 0, got {r}")
    return -G0 * M / r * (1.0 + params.epsilon * (r/params.ell_s)**params.beta)

def phi_rg_improved(r: float, M: float, params: RGImprovedParams) -> float:
    """
    RG-improved potential:
        Phi(r) = - G(r) M / r, where G(r) = G0[1 + xi*(1 + r/r0)^{-n}]
    """
    if r <= 0.0:
        raise ValueError(f"r must be > 0, got {r}")
    G_r = G0 * (1.0 + params.xi * (1.0 + r/params.r0)**(-params.n))
    return -G_r * M / r

def _richardson_derivative(f: Callable[[float], float], x: float, h: float, order: int = 2) -> float:
    """
    Derivative f'(x) using centered differences with Richardson extrapolation.
    """
    if h <= 0.0:
        raise ValueError("h must be positive")

    D = []
    for k in range(order + 1):
        hk = h / (2.0 ** k)
        Dk = (f(x + hk) - f(x - hk)) / (2.0 * hk)
        D.append(Dk)

    for m in range(1, order + 1):
        for k in range(order, m - 1, -1):
            D[k] = D[k] + (D[k] - D[k - 1]) / (4 ** m - 1)
    return D[order]

# -------------------------
# Perihelion precession
# -------------------------

def perihelion_precession_yukawa(a: float, e: float, M: float, params: YukawaParams, h_e: float = 1e-4) -> float:
    """
    Yukawa correction to perihelion precession (Δϖ_Yukawa) relative to Newtonian potential.
    Returns radians per orbit.
    """
    if not (0.0 <= e < 1.0):
        raise ValueError(f"Eccentricity must be in [0,1), got e={e}")

    R_s = 2.0 * G0 * M / c0**2
    if a <= 5.0 * R_s:
        raise ValueError("Semi-major axis too close to Schwarzschild radius; weak-field approximation breaks down.")

    def avg_delta_phi(ecc: float) -> float:
        def integrand(f):
            r = a * (1.0 - ecc**2) / (1.0 + ecc * np.cos(f))
            phiY = phi_yukawa(r, M, params)
            phiN = -G0 * M / r
            return phiY - phiN
        res, _ = quad(integrand, 0.0, 2.0 * np.pi, epsabs=0.0, epsrel=1e-8)
        return res / (2.0 * np.pi)

    def f_e(ecc: float) -> float:
        return avg_delta_phi(ecc)

    n = np.sqrt(G0 * M / a**3)
    h_loc = min(h_e, 0.1 * max(e, 1e-3))

    if e < 1e-3:
        e_plus = e + h_loc
        e_minus = max(e - h_loc, 0.0)
        d_avg_de = (f_e(e_plus) - f_e(e_minus)) / (e_plus - e_minus)
    else:
        d_avg_de = _richardson_derivative(f_e, e, h_loc, order=2)

    Delta_varpi = -(1.0 / (n * a**2 * e)) * d_avg_de
    return Delta_varpi

# -------------------------
# Light deflection with systematics
# -------------------------

def deflection_angle_yukawa(b: float, M: float, params: YukawaParams, gamma_inf: float = 1.0) -> Tuple[float, float]:
    """
    Deflection angle for light ray with impact parameter b in minimal ansatz (Psi = gamma_inf * Phi).
    Returns (alpha_total, alpha_GR) in radians.
    """
    if b <= 0.0:
        raise ValueError("Impact parameter b must be > 0")

    alpha_gr = 4.0 * G0 * M / (c0**2 * b)

    def phi_plus_psi(r: float) -> float:
        phi = phi_yukawa(r, M, params)
        psi = gamma_inf * phi
        return phi + psi

    def integrand(z: float) -> float:
        r = np.sqrt(b**2 + z**2)
        h_val = 1e-3 * max(r, b, R_SUN)
        dPhi_dR = (phi_plus_psi(r + h_val) - phi_plus_psi(r - h_val)) / (2.0 * h_val)
        return dPhi_dR * b / r

    z_max = 1e11
    res, _ = quad(integrand, -z_max, z_max, epsabs=0.0, epsrel=1e-6)
    alpha_tot = res / c0**2
    return alpha_tot, alpha_gr

def deflection_solar_j2(b: float, phi: float) -> float:
    """
    Deflection due to solar quadrupole moment J2.
    phi: angle between impact parameter vector and solar rotation axis [rad]
    """
    alpha_gr = 4.0 * G0 * M_SUN / (c0**2 * b)
    return J2_SUN * (R_SUN**2 / b**2) * alpha_gr * (1.5 * np.cos(2*phi) - 0.5)

def deflection_angle_with_systematics(b: float, phi: float, params: YukawaParams,
                                    include_j2: bool = True, gamma_inf: float = 1.0) -> Dict[str, float]:
    """
    Comprehensive deflection angle calculation including systematics.
    """
    alpha_yukawa, alpha_gr = deflection_angle_yukawa(b, M_SUN, params, gamma_inf)

    results = {
        'yukawa': alpha_yukawa,
        'GR': alpha_gr,
        'j2': 0.0,
        'total': alpha_yukawa
    }

    if include_j2:
        j2_effect = deflection_solar_j2(b, phi)
        results['j2'] = j2_effect
        results['total'] = alpha_yukawa + j2_effect

    return results

# -------------------------
# Effective energy density - VERSIÓN CORREGIDA
# -------------------------

def rho_eff_yukawa_r(r: float, M: float, params: YukawaParams) -> float:
    """
    Effective energy density rho_eff(r) in weak-field limit using ANALYTIC expression.

    For Yukawa potential: Φ(r) = - (GM/r)[1 + α exp(-r/λ)]
    The Laplacian gives: ∇²Φ = (GMα/(rλ²)) exp(-r/λ)
    Therefore: ρ_eff = ∇²Φ/(4πG) = (Mα/(4πrλ²)) exp(-r/λ)

    This is the EXACT analytic expression, more stable than numerical differentiation.
    """
    if r <= 0.0:
        raise ValueError("r must be > 0")

    # Analytic expression for Yukawa effective density
    if abs(params.alpha) < 1e-15:  # GR case
        return 0.0

    prefactor = M * params.alpha / (4 * np.pi * r * params.lambda_m**2)
    return prefactor * np.exp(-r / params.lambda_m)

def rho_eff_gr_r(r: float, M: float) -> float:
    """
    Effective energy density for GR case (point mass).
    In GR, outside the mass: ρ_eff = 0, but we include the point mass source term
    for comparison purposes.
    """
    if r <= 0.0:
        raise ValueError("r must be > 0")

    # For a point mass in GR, ρ_eff = M * δ³(r) = 0 for r > 0
    return 0.0

# -------------------------
# ppE framework for gravitational waves
# -------------------------

def ppe_yukawa_coefficients(m1: float, m2: float, params: YukawaParams, f_orb: float) -> Dict[str, float]:
    """
    Compute ppE coefficients for Yukawa correction in inspiraling binary.

    Parameters:
    m1, m2: component masses [kg]
    params: YukawaParams instance
    f_orb: orbital frequency [Hz]

    Returns:
    dict with ppE coefficients {beta_ppe, b_ppe}
    """
    M = m1 + m2
    mu = m1 * m2 / M
    eta = mu / M

    r_orb = (G0 * M / (np.pi * f_orb)**2)**(1/3)
    x = r_orb / params.lambda_m

    # Binding energy correction factor for circular orbits
    F_x = np.exp(-x) * (1 + x + x**2/3)

    beta_ppe = -3/(128 * eta**(5/3)) * params.alpha * F_x
    b_ppe = -5/3

    return {'beta_ppe': beta_ppe, 'b_ppe': b_ppe}

def ppe_power_law_coefficients(m1: float, m2: float, params: PowerLawParams) -> Dict[str, float]:
    """
    Compute ppE coefficients for power-law correction.
    """
    M = m1 + m2
    mu = m1 * m2 / M
    eta = mu / M

    b_ppe = (2 * params.beta - 8) / 3
    beta_ppe = (3 / (128 * eta**(5/3))) * params.epsilon * (G0 * M / (c0**2 * params.ell_s))**params.beta

    return {'beta_ppe': beta_ppe, 'b_ppe': b_ppe}

# -------------------------
# Bayesian analysis tools
# -------------------------

def compute_bayes_factor(log_evidence_model1: float, log_evidence_model2: float) -> float:
    """
    Compute Bayes factor from log evidences.

    Returns:
    log_B12: log(Bayes factor) favoring model1 over model2
    """
    return log_evidence_model1 - log_evidence_model2

def jeffreys_scale(bayes_factor: float) -> str:
    """
    Interpret Bayes factor according to Jeffreys' scale.
    """
    if bayes_factor < 1:
        return "Negative (supports M2)"
    elif 1 <= bayes_factor < 3:
        return "Barely worth mentioning"
    elif 3 <= bayes_factor < 10:
        return "Substantial"
    elif 10 <= bayes_factor < 30:
        return "Strong"
    elif 30 <= bayes_factor < 100:
        return "Very strong"
    else:
        return "Decisive"

def fisher_matrix(observables_func: Callable, params: np.ndarray, data_covariance: np.ndarray,
                 delta: float = 1e-6) -> np.ndarray:
    """
    Compute Fisher information matrix for gravity parameters.

    Parameters:
    observables_func: function computing observables from parameters
    params: parameter values
    data_covariance: covariance matrix of experimental data
    delta: finite difference step

    Returns:
    F_ij: Fisher matrix
    """
    n_params = len(params)
    n_obs = data_covariance.shape[0]

    # Get central value
    obs_central = observables_func(params)

    # Compute derivatives numerically
    derivatives = np.zeros((n_obs, n_params))

    for i in range(n_params):
        params_plus = params.copy()
        params_plus[i] += delta
        obs_plus = observables_func(params_plus)

        derivatives[:, i] = (obs_plus - obs_central) / delta

    # Fisher matrix
    F = derivatives.T @ np.linalg.inv(data_covariance) @ derivatives
    return F

# -------------------------
# Stability and energy conditions - VERSIÓN CORREGIDA
# -------------------------

def check_energy_conditions_yukawa(r: float, M: float, params: YukawaParams) -> Dict[str, bool]:
    """
    Check energy conditions for Yukawa ansatz in weak-field approximation.

    Returns:
    Dictionary with WEC, DEC, SEC, NEC satisfaction flags
    """
    rho_eff = rho_eff_yukawa_r(r, M, params)

    # In weak-field, for Yukawa potential, the effective density
    # is the ONLY contribution outside the source
    p_radial = 0.0
    p_tangential = 0.0

    # Weak Energy Condition (WEC): ρ ≥ 0
    wec = rho_eff >= 0

    # Null Energy Condition (NEC): ρ + p_i ≥ 0
    nec = (rho_eff + p_radial >= 0) and (rho_eff + p_tangential >= 0)

    # Dominant Energy Condition (DEC): ρ ≥ |p_i|
    dec = (rho_eff >= abs(p_radial)) and (rho_eff >= abs(p_tangential))

    # Strong Energy Condition (SEC): ρ + ∑ p_i ≥ 0 and ρ + p_i ≥ 0
    sec = (rho_eff + p_radial + 2*p_tangential >= 0) and nec

    return {
        'WEC': wec,
        'NEC': nec,
        'DEC': dec,
        'SEC': sec
    }

def stability_timescale_yukawa(params: YukawaParams) -> float:
    """
    Estimate stability timescale for Yukawa corrections.
    For alpha_Y < 0, tachyonic instabilities may develop.
    """
    if params.alpha >= 0:
        return np.inf  # Stable

    # Rough estimate: tau ~ lambda/(c*sqrt|alpha|)
    return params.lambda_m / (c0 * np.sqrt(abs(params.alpha)))

# -------------------------
# Multi-probe constraint analysis
# -------------------------

class MultiProbeAnalyzer:
    """
    Analyze constraints across multiple experimental probes.
    """

    def __init__(self):
        self.probes = {}

    def add_probe(self, name: str, likelihood_func: Callable, scale: float):
        """
        Add an experimental probe.

        Parameters:
        name: probe identifier
        likelihood_func: function computing log-likelihood
        scale: characteristic length scale [m]
        """
        self.probes[name] = {
            'likelihood': likelihood_func,
            'scale': scale
        }

    def combined_likelihood(self, params: np.ndarray) -> float:
        """
        Compute combined log-likelihood across all probes.
        """
        total_loglike = 0.0
        for probe in self.probes.values():
            total_loglike += probe['likelihood'](params)
        return total_loglike

    def fisher_matrix_combined(self, params: np.ndarray, data_covariances: Dict[str, np.ndarray]) -> np.ndarray:
        """
        Compute combined Fisher matrix across probes.
        """
        total_fisher = np.zeros((len(params), len(params)))

        for name, probe in self.probes.items():
            def obs_func(p):
                # Placeholder - should be implemented for specific probes
                return np.array([perihelion_precession_yukawa(0.387*AU, 0.2056, M_SUN,
                                 YukawaParams(alpha=p[0], lambda_m=p[1]))])

            F_probe = fisher_matrix(obs_func, params, data_covariances[name])
            total_fisher += F_probe

        return total_fisher

# -------------------------
# Plotting functions
# -------------------------

def make_yukawa_constraints_plot(filename: str = "yukawa_constraints.pdf") -> None:
    """
    Schematic combined constraints in (lambda_Y, alpha_Y) plane.
    """
    lam = np.logspace(-4, 2, 200)  # in AU

    alpha_cassini = 2.3e-5 * np.ones_like(lam)
    alpha_mercury = 3.0e-4 * np.ones_like(lam)
    alpha_torsion = 2e-3 * (lam / 1e-4) ** 0.5
    alpha_gaia = 1e-4 * np.ones_like(lam)
    alpha_et = 1e-4 * (lam / 1e-1) ** (-0.5)  # Einstein Telescope forecast

    plt.figure(figsize=(7.0, 5.5))
    plt.loglog(lam, alpha_cassini, label="Cassini (γ)", ls="--", linewidth=2)
    plt.loglog(lam, alpha_mercury, label="Mercury perihelion", ls="-.", lw=1.8)
    plt.loglog(lam, alpha_torsion, label="Torsion balance", ls=":", lw=1.8)
    plt.loglog(lam, alpha_gaia, label="Gaia-like forecast", ls="-", linewidth=2)
    plt.loglog(lam, alpha_et, label="ET forecast", ls="--", linewidth=2, alpha=0.7)

    plt.xlabel(r"$\lambda_{\rm Y}\,[\mathrm{AU}]$", fontsize=12)
    plt.ylabel(r"$|\alpha_{\rm Y}|$", fontsize=12)
    plt.ylim(1e-6, 1e0)
    plt.xlim(1e-4, 1e2)
    plt.grid(True, which="both", ls=":", alpha=0.7)
    plt.legend(fontsize=10)
    plt.title("Multi-probe constraints on Yukawa parameters", fontsize=12)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

def plot_rho_eff_yukawa(filename: str = "rho_eff_yukawa.pdf") -> None:
    """
    Plot ratio |rho_eff(r)|/|rho_GR(r)| for representative Yukawa parameters.
    Uses ANALYTIC expressions for better stability.
    """
    M = M_SUN
    r_grid = np.logspace(-2, 1, 200) * AU
    alphas = [-1e-4, 0.0, 1e-4]
    colors = ["C0", "C1", "C2"]
    lambda_m = 0.1 * AU

    plt.figure(figsize=(7.0, 5.5))

    for alpha, col in zip(alphas, colors):
        params = YukawaParams(alpha=alpha, lambda_m=lambda_m)

        # Compute ratios using analytic expressions
        ratio = []
        for r in r_grid:
            if abs(alpha) < 1e-15:  # GR case
                ratio.append(1.0)
            else:
                rho_eff = rho_eff_yukawa_r(r, M, params)
                rho_gr = 0.0  # In GR, outside source ρ_eff = 0

                # Handle the case where GR density is zero
                if abs(rho_gr) < 1e-40:
                    # For visualization, we normalize by the Yukawa density magnitude
                    # at a reference point to show the relative behavior
                    rho_ref = M * abs(alpha) / (4 * np.pi * (0.1*AU) * lambda_m**2) * np.exp(-1)
                    ratio_val = abs(rho_eff) / rho_ref if rho_ref > 0 else 1.0
                else:
                    ratio_val = abs(rho_eff) / abs(rho_gr)

                ratio.append(ratio_val)

        plt.loglog(r_grid / AU, np.abs(ratio),
                  label=rf"$\alpha_{{\rm Y}}={alpha:.4f}$", color=col, linewidth=2)

    plt.xlabel(r"$r\,[\mathrm{AU}]$", fontsize=12)
    plt.ylabel(r"$|\rho_{\mathrm{eff}}(r)/\rho_{\mathrm{GR}}(r)|$", fontsize=12)
    plt.xlim(1e-2, 1e1)
    plt.ylim(1e-1, 1e4)
    plt.grid(True, which="both", ls=":", alpha=0.7)
    plt.legend(fontsize=10)
    plt.title("Effective density ratio for Yukawa corrections", fontsize=12)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

def plot_systematic_effects(filename: str = "systematics_analysis.pdf") -> None:
    """
    Plot systematic effects for Gaia-like tests.
    """
    b_values = np.logspace(0, 1, 100) * R_SUN  # 1-10 R_sun
    phi = np.pi/4  # Example angle

    # Yukawa effect for reference
    params_ref = YukawaParams(alpha=1e-4, lambda_m=0.1*AU)

    plt.figure(figsize=(8, 6))

    # Yukawa deflection
    alpha_yukawa = [deflection_angle_yukawa(b, M_SUN, params_ref)[0] for b in b_values]
    alpha_gr = [4.0 * G0 * M_SUN / (c0**2 * b) for b in b_values]
    yukawa_effect = [(ay - ag)/ag for ay, ag in zip(alpha_yukawa, alpha_gr)]

    # J2 effect
    j2_effect = [deflection_solar_j2(b, phi)/ag for b, ag in zip(b_values, alpha_gr)]

    plt.semilogx(b_values/R_SUN, np.array(yukawa_effect)*100,
                label='Yukawa correction ($\\alpha=10^{-4}$)', linewidth=2)
    plt.semilogx(b_values/R_SUN, np.array(j2_effect)*100,
                label='Solar $J_2$ effect', linewidth=2, linestyle='--')

    plt.xlabel(r'Impact parameter $b/R_\odot$', fontsize=12)
    plt.ylabel('Relative effect on deflection [%]', fontsize=12)
    plt.grid(True, which="both", ls=":", alpha=0.7)
    plt.legend(fontsize=10)
    plt.title('Systematic effects in light deflection measurements', fontsize=12)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

# -------------------------
# Validation tests - VERSIÓN CORREGIDA
# -------------------------

def test_yukawa_zero_limit() -> None:
    """Test that Yukawa correction vanishes when alpha -> 0."""
    a = 0.387 * AU
    e = 0.2056
    M = M_SUN
    params = YukawaParams(alpha=0.0, lambda_m=1.0 * AU)
    DeltaY = perihelion_precession_yukawa(a, e, M, params)
    tol = 1e-12
    print(f"[test_yukawa_zero_limit] |DeltaY| = {abs(DeltaY):.3e} rad")
    assert abs(DeltaY) < tol

def test_light_deflection_GR_limit() -> None:
    """Test recovery of GR deflection when alpha->0 and gamma_inf=1."""
    b = R_SUN
    M = M_SUN
    params = YukawaParams(alpha=0.0, lambda_m=1.0 * AU)
    alpha_tot, alpha_gr = deflection_angle_yukawa(b, M, params, gamma_inf=1.0)
    rel_err = abs(alpha_tot - alpha_gr) / abs(alpha_gr)
    print(f"[test_light_deflection_GR_limit] relative error = {rel_err:.3e}")
    assert rel_err < 1e-2

def test_ppe_coefficients() -> None:
    """Test ppE coefficient calculations."""
    m1 = 1.4 * M_SUN  # Neutron star
    m2 = 1.4 * M_SUN
    params = YukawaParams(alpha=1e-4, lambda_m=100 * AU)
    f_orb = 0.01  # Hz

    coeffs = ppe_yukawa_coefficients(m1, m2, params, f_orb)
    print(f"[test_ppe_coefficients] beta_ppe = {coeffs['beta_ppe']:.3e}, b_ppe = {coeffs['b_ppe']:.3f}")
    assert abs(coeffs['b_ppe'] - (-5/3)) < 1e-10

def test_energy_conditions() -> None:
    """Test energy condition checks."""
    r = 1.0 * AU
    M = M_SUN
    params = YukawaParams(alpha=1e-4, lambda_m=0.1 * AU)

    conditions = check_energy_conditions_yukawa(r, M, params)
    print(f"[test_energy_conditions] WEC: {conditions['WEC']}")

    # For alpha > 0, the analytic expression should give rho_eff > 0
    rho_eff = rho_eff_yukawa_r(r, M, params)
    print(f"[test_energy_conditions] rho_eff = {rho_eff:.3e} kg/m³")

    # The test should pass now with the analytic expression
    assert conditions['WEC'] == True, f"WEC failed with rho_eff = {rho_eff}"

def run_all_tests() -> None:
    """Run all validation tests."""
    print("Running validation tests...")
    test_yukawa_zero_limit()
    test_light_deflection_GR_limit()
    test_ppe_coefficients()
    test_energy_conditions()
    print("All tests passed!")

# -------------------------
# Example usage and main
# -------------------------

def example_analysis():
    """Demonstrate the expanded toolkit capabilities."""

    print("=== Yukawa Gravity Toolkit Example Analysis ===")

    # 1. Basic Yukawa effects
    params = YukawaParams(alpha=1e-4, lambda_m=0.1*AU)
    a_merc = 0.387 * AU
    e_merc = 0.2056

    delta_varpi = perihelion_precession_yukawa(a_merc, e_merc, M_SUN, params)
    print(f"Mercury perihelion precession: {delta_varpi:.3e} rad/orbit")

    # 2. Light deflection with systematics
    b = 2 * R_SUN
    phi = np.pi/4
    deflection_results = deflection_angle_with_systematics(b, phi, params, include_j2=True)
    print(f"Deflection at 2 R_sun: {deflection_results['total']*206265:.2f} arcsec")
    print(f"  - Yukawa contribution: {deflection_results['yukawa']*206265:.2f} arcsec")
    print(f"  - J2 contribution: {deflection_results['j2']*206265:.4f} arcsec")

    # 3. ppE coefficients for GWs
    m1 = 30 * M_SUN
    m2 = 30 * M_SUN
    f_orb = 0.01
    ppe_coeffs = ppe_yukawa_coefficients(m1, m2, params, f_orb)
    print(f"ppE coefficients: beta = {ppe_coeffs['beta_ppe']:.3e}, b = {ppe_coeffs['b_ppe']:.3f}")

    # 4. Energy conditions
    r_test = 1.0 * AU
    energy_conds = check_energy_conditions_yukawa(r_test, M_SUN, params)
    print("Energy conditions at 1 AU:")
    for cond, value in energy_conds.items():
        print(f"  {cond}: {value}")

    # 5. Stability analysis
    stability_time = stability_timescale_yukawa(params)
    if np.isfinite(stability_time):
        print(f"Stability timescale: {stability_time:.3e} s")
    else:
        print("Configuration is stable")

    print("=== Example analysis complete ===")

# -------------------------
# Main execution
# -------------------------

if __name__ == "__main__":
    # Run validation tests
    run_all_tests()

    # Generate plots
    print("Generating constraint plots...")
    make_yukawa_constraints_plot("yukawa_constraints.pdf")
    plot_rho_eff_yukawa("rho_eff_yukawa.pdf")
    plot_systematic_effects("systematics_analysis.pdf")

    # Run example analysis
    example_analysis()

    print("All tasks completed successfully!")

Running validation tests...
[test_yukawa_zero_limit] |DeltaY| = 0.000e+00 rad
[test_light_deflection_GR_limit] relative error = 2.320e-05
[test_ppe_coefficients] beta_ppe = -2.362e-05, b_ppe = -1.667
[test_energy_conditions] WEC: True
[test_energy_conditions] rho_eff = 2.146e-11 kg/m³
All tests passed!
Generating constraint plots...
=== Yukawa Gravity Toolkit Example Analysis ===
Mercury perihelion precession: 3.685e-11 rad/orbit
Deflection at 2 R_sun: 0.88 arcsec
  - Yukawa contribution: 0.88 arcsec
  - J2 contribution: -0.0000 arcsec
ppE coefficients: beta = -2.362e-05, b = -1.667
Energy conditions at 1 AU:
  WEC: True
  NEC: True
  DEC: True
  SEC: True
Configuration is stable
=== Example analysis complete ===
All tasks completed successfully!
